# Sommelier Kaggle Full Run

Notebook này chạy pipeline chính của branch `codex/kaggle-main-html-viewer` trên Kaggle. Nó tự clone repo, chuẩn hóa audio input, chạy `main_original_ASR_MoE.py`, gom JSON/MP3 về `/kaggle/working/run_full`, và tạo file zip artifact thật để tải xuống.

Điều kiện trước khi chạy:
- Kaggle Accelerator: GPU bật.
- Kaggle Internet: bật.
- Kaggle Secret có `HF_TOKEN`.
- Dataset audio đã add vào notebook. Notebook sẽ tự tìm file audio đầu tiên trong `/kaggle/input`.


## 0. Cấu hình run

In [ ]:
REPO_URL = "https://github.com/lamkdhe180931-arch/sommelier.git"
BRANCH = "codex/kaggle-main-html-viewer"

RUN_DIR = "/kaggle/working/run_full"
INPUT_DIR = f"{RUN_DIR}/00_input"
EXPORT_DIR = f"{RUN_DIR}/05_export"
FINAL_DIR = f"{EXPORT_DIR}/final"
DATA_AUDIO_DIR = f"{FINAL_DIR}/data_audio"
PREVIEW_DIR = f"{RUN_DIR}/preview"
LOG_DIR_PATH = f"{RUN_DIR}/logs"
AUDIO_WAV = f"{INPUT_DIR}/full.wav"

# Để None nếu muốn chạy full audio. Để 300 nếu muốn test nhanh 5 phút.
AUDIO_LIMIT_SECONDS = 300

# Bật/tắt các bước nặng.
INSTALL_DEPS = True
RUN_DEMUCS = True
RUN_SEPREFORMER = True
ASR_MOE = True

# ASRMoE trên Kaggle 2xT4: Whisper GPU0, PhoWhisper/ChunkFormer GPU1. Cell chạy pipeline sẽ tự chuyển index 1 về 0 nếu Kaggle chỉ cấp 1 GPU.
WHISPER_DEVICE_INDEX = 0
ASR_MOE_DEVICE_INDEX = 1
PHOWHISPER_DEVICE_INDEX = 1
CTC_DEVICE_INDEX = 1
PANNS_DEVICE_INDEX = 1
DEMUCS_DEVICE_INDEX = 1
SEPREFORMER_DEVICE_INDEX = 1

WHISPER_ARCH = "large-v3"
ASR_LANGUAGE = "vi"
PHOWHISPER_MODEL_NAME = "vinai/PhoWhisper-large"
CTC_MODEL_NAME = "khanhld/chunkformer-ctc-large-vie"
COMPUTE_TYPE = "float16"
ASR_THREADS = 4

OVERLAP_THRESHOLD = 0.2
SPEAKER_LINK_THRESHOLD = 0.75
ASR_QUALITY_GUARD = True
ASR_MICRO_SEGMENT_SECONDS = 0.5
ASR_SHORT_SEGMENT_SECONDS = 1.0
ASR_VI_AGREEMENT_THRESHOLD = 0.75

HF_SECRET_NAME = "HF_TOKEN"


In [ ]:
from pathlib import Path
import os
import shlex
import subprocess

LOG_DIR = Path(LOG_DIR_PATH)
for _dir in [INPUT_DIR, EXPORT_DIR, FINAL_DIR, DATA_AUDIO_DIR, PREVIEW_DIR, LOG_DIR_PATH]:
    Path(_dir).mkdir(parents=True, exist_ok=True)


def _format_cmd(cmd):
    if isinstance(cmd, (list, tuple)):
        return " ".join(shlex.quote(str(part)) for part in cmd)
    return str(cmd)


def tail_file(path, n=30):
    path = Path(path)
    if not path.exists():
        return ""
    lines = path.read_text(encoding="utf-8", errors="replace").splitlines()
    return "\n".join(lines[-n:])


def run_logged(cmd, log_name, cwd=None, env=None, shell=False, tail=20):
    log_path = LOG_DIR / log_name
    cwd = cwd or os.getcwd()
    print("Running:", _format_cmd(cmd))
    print("Log:", log_path)
    with open(log_path, "w", encoding="utf-8", errors="replace") as log:
        proc = subprocess.run(
            cmd,
            cwd=cwd,
            env=env,
            shell=shell,
            stdout=log,
            stderr=subprocess.STDOUT,
            text=True,
        )
    print("Exit code:", proc.returncode)
    if tail:
        log_tail = tail_file(log_path, n=tail)
        if log_tail:
            print(f"--- last {tail} log lines ---")
            print(log_tail)
    if proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd)
    return log_path


def export_audio_preview(audio_segment, out_path, seconds=30):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    audio_segment[: int(seconds * 1000)].export(out_path, format="wav")
    return out_path


## 1. Clone repo

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

os.chdir("/kaggle/working")
repo_dir = Path("/kaggle/working/sommelier")
if repo_dir.exists():
    shutil.rmtree(repo_dir)

run_logged(["git", "clone", "-b", BRANCH, REPO_URL, str(repo_dir)], "01_clone_repo.log", cwd="/kaggle/working", tail=30)
os.chdir(repo_dir / "podcast-pipeline")
print("cwd:", os.getcwd())
print("branch:", subprocess.check_output(["git", "rev-parse", "--abbrev-ref", "HEAD"], text=True).strip())
print("commit:", subprocess.check_output(["git", "log", "-1", "--oneline"], text=True).strip())


## 2. Cài dependencies

In [ ]:
import os
from pathlib import Path

os.chdir("/kaggle/working/sommelier/podcast-pipeline")

if INSTALL_DEPS:
    run_logged(["apt-get", "update", "-y"], "02_apt_update.log", tail=10)
    run_logged(["apt-get", "install", "-y", "ffmpeg", "git", "git-lfs"], "03_apt_install.log", tail=10)
    run_logged(["python", "-m", "pip", "install", "-U", "pip", "setuptools", "wheel", "packaging", "ninja"], "04_pip_base.log", tail=12)

    req = Path("requirements.txt").read_text(encoding="utf-8")
    skip_prefixes = (
        "nemo-toolkit[all]",
        "torch==",
        "torchaudio==",
        "torchvision==",
        "triton==",
        "nvidia-",
    )
    filtered = []
    for line in req.splitlines():
        stripped = line.strip()
        if not stripped or stripped.startswith("#"):
            filtered.append(line)
            continue
        if any(stripped.startswith(prefix) for prefix in skip_prefixes):
            continue
        filtered.append(line)
    Path("requirements-kaggle.txt").write_text("\n".join(filtered) + "\n", encoding="utf-8")

    run_logged(["python", "-m", "pip", "install", "-r", "requirements-kaggle.txt"], "05_pip_requirements.log", tail=25)
    run_logged(["python", "-m", "pip", "uninstall", "-y", "nemo-toolkit", "lightning", "pytorch-lightning"], "06_pip_uninstall_nemo.log", tail=8)
    run_logged(["python", "-m", "pip", "install", "lightning==2.4.0", "pytorch-lightning==2.5.2"], "07_pip_lightning.log", tail=12)
    run_logged(["python", "-m", "pip", "install", "nemo-toolkit[asr]==2.4.0"], "08_pip_nemo_asr.log", tail=25)
    run_logged([
        "python", "-m", "pip", "install", "--no-cache-dir", "--force-reinstall",
        "torch==2.7.1", "torchaudio==2.7.1", "torchvision==0.22.1",
        "--index-url", "https://download.pytorch.org/whl/cu126",
    ], "09_pip_torch_stack.log", tail=25)
    run_logged(["python", "-m", "pip", "install", "pillow<12.0"], "10_pip_pillow.log", tail=8)
    run_logged(["python", "-m", "pip", "install", "--no-cache-dir", "--force-reinstall", "--no-deps", "torchmetrics==1.7.4"], "11_pip_torchmetrics.log", tail=8)
    run_logged([
        "python", "-m", "pip", "install", "--no-cache-dir", "--force-reinstall",
        "numpy==2.2.6", "numba==0.61.2", "llvmlite==0.44.0",
    ], "12_pip_numpy_numba.log", tail=12)
    run_logged([
        "python", "-m", "pip", "install", "--no-cache-dir", "--no-deps",
        "chunkformer==1.2.2", "colorama==0.4.6",
    ], "13_pip_chunkformer.log", tail=20)
else:
    print("INSTALL_DEPS=False, bỏ qua cài dependencies")

print("Dependency install logs saved in:", LOG_DIR)


## 3. Kiểm tra môi trường

In [ ]:
import importlib.metadata as importlib_metadata
import numpy, numba, torch

def version_or_missing(package):
    try:
        return importlib_metadata.version(package)
    except importlib_metadata.PackageNotFoundError:
        return "missing"

print("nemo-toolkit:", version_or_missing("nemo-toolkit"))
print("chunkformer:", version_or_missing("chunkformer"))

from chunkformer import ChunkFormerModel
print("chunkformer import ok:", ChunkFormerModel.__name__)
print("numpy:", numpy.__version__)
print("numba:", numba.__version__)
print("torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count() if torch.cuda.is_available() else 0)
print("GPU 0:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

import whisperx
print("whisperx ok")

import nemo.collections.asr as nemo_asr
print("nemo asr ok")

from nemo.collections.asr.models import SortformerEncLabelModel
print("sortformer import ok")


## 4. Gắn Hugging Face token vào config

In [ ]:
import json
from kaggle_secrets import UserSecretsClient
from huggingface_hub import whoami

token = UserSecretsClient().get_secret(HF_SECRET_NAME)
print("HF token:", token[:8] + "..." if token else "missing")
print(whoami(token=token))

with open("config.json", "r", encoding="utf-8") as f:
    cfg = json.load(f)

cfg["huggingface_token"] = token
cfg["entrypoint"]["input_folder_path"] = INPUT_DIR

with open("config.json", "w", encoding="utf-8") as f:
    json.dump(cfg, f, indent=2, ensure_ascii=False)

print("config.json updated")


## 5. Tìm audio input và chuẩn hóa audio

In [ ]:
from pathlib import Path

audio_exts = {".mp3", ".wav", ".m4a", ".flac", ".aac", ".ogg", ".opus"}
audio_candidates = sorted(
    p for p in Path("/kaggle/input").rglob("*")
    if p.is_file() and p.suffix.lower() in audio_exts
)

if not audio_candidates:
    raise FileNotFoundError("Không tìm thấy audio trong /kaggle/input. Hãy Add Input hoặc Upload audio trước.")

AUDIO_IN = str(audio_candidates[0])
print("AUDIO_IN:", AUDIO_IN)
print("AUDIO_WAV:", AUDIO_WAV)
print("RUN_DIR:", RUN_DIR)

Path(INPUT_DIR).mkdir(parents=True, exist_ok=True)
Path(RUN_DIR).mkdir(parents=True, exist_ok=True)

cmd = ["ffmpeg", "-hide_banner", "-y", "-i", AUDIO_IN]
if AUDIO_LIMIT_SECONDS:
    cmd += ["-t", str(AUDIO_LIMIT_SECONDS)]
cmd += ["-ac", "1", "-ar", "16000", AUDIO_WAV]
run_logged(cmd, "00_prepare_audio_ffmpeg.log", cwd="/kaggle/working", tail=15)


In [ ]:
from pathlib import Path
from pydub import AudioSegment
from IPython.display import Audio, display

audio = AudioSegment.from_file(AUDIO_WAV)
print("Audio:", AUDIO_WAV)
print("Duration seconds:", len(audio) / 1000)
print("Frame rate:", audio.frame_rate)
print("Channels:", audio.channels)

preview_path = export_audio_preview(audio, Path(PREVIEW_DIR) / "preview_input_30s.wav", seconds=30)
print("Preview first 30s:", preview_path)
display(Audio(str(preview_path)))


## 6. Tải model phụ

In [ ]:
from huggingface_hub import hf_hub_download

if RUN_DEMUCS:
    panns_path = hf_hub_download(
        repo_id="thelou1s/panns-inference",
        filename="Cnn14_mAP=0.431.pth",
        local_dir="/kaggle/working/sommelier/panns_data",
    )
    print("PANNs checkpoint:", panns_path)
else:
    print("RUN_DEMUCS=False, bỏ qua tải PANNs")


In [ ]:
import os
from pathlib import Path

if RUN_SEPREFORMER:
    run_logged(["git", "lfs", "install"], "13_git_lfs_install.log", cwd="/kaggle/working/sommelier", tail=10)
    os.chdir("/kaggle/working/sommelier")
    if not Path("SepReformer").exists():
        run_logged(["git", "clone", "https://github.com/dmlguq456/SepReformer.git", "SepReformer"], "14_clone_sepreformer.log", cwd="/kaggle/working/sommelier", tail=20)
    run_logged(["git", "lfs", "pull"], "15_sepreformer_lfs_pull.log", cwd="/kaggle/working/sommelier/SepReformer", tail=20)
    run_logged([
        "python", "-m", "pip", "install", "--no-deps",
        "mir-eval==0.7", "ptflops==0.7.4", "thop==0.1.1.post2209072238", "torchinfo==1.8.0",
    ], "16_sepreformer_extra_deps.log", cwd="/kaggle/working/sommelier/SepReformer", tail=12)

    log = Path("/kaggle/working/sommelier/SepReformer/models/SepReformer_Base_WSJ0/log")
    src = log / "scratch_weight"
    dst = log / "scratch_weights"
    if src.exists() and not dst.exists():
        os.symlink(src, dst)

    ckpts = list(log.rglob("*.pt")) + list(log.rglob("*.pth"))
    print("SepReformer checkpoints:", len(ckpts))
    for p in ckpts[:10]:
        print(p)
else:
    print("RUN_SEPREFORMER=False, bỏ qua SepReformer")

os.chdir("/kaggle/working/sommelier/podcast-pipeline")


## 7. Cài cuDNN 8 riêng cho faster-whisper/ctranslate2

In [ ]:
import shutil
from pathlib import Path

cudnn_dir = Path("/kaggle/working/cudnn8")
if cudnn_dir.exists():
    shutil.rmtree(cudnn_dir)

run_logged([
    "python", "-m", "pip", "install", "--target", str(cudnn_dir),
    "nvidia-cudnn-cu12==8.9.7.29",
], "17_pip_cudnn8.log", cwd="/kaggle/working/sommelier/podcast-pipeline", tail=20)

matches = sorted(cudnn_dir.rglob("libcudnn_ops_infer.so.8"))
print("libcudnn matches:", len(matches))
for p in matches[:5]:
    print(p)


## 8. Chạy full pipeline trên branch hiện tại

In [ ]:
import os
import subprocess
import torch

os.chdir("/kaggle/working/sommelier/podcast-pipeline")
subprocess.run(["nvidia-smi"], check=False)

env = os.environ.copy()
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
extra_ld_paths = [
    "/kaggle/working/cudnn8/nvidia/cudnn/lib",
    "/kaggle/working/cudnn8/nvidia/cublas/lib",
    "/kaggle/working/cudnn8/nvidia/cuda_nvrtc/lib",
]
env["LD_LIBRARY_PATH"] = ":".join(extra_ld_paths + [env.get("LD_LIBRARY_PATH", "")]).rstrip(":")

VISIBLE_GPU_COUNT = torch.cuda.device_count() if torch.cuda.is_available() else 0
print("Visible CUDA device count:", VISIBLE_GPU_COUNT)


def _clamp_device_index(name, requested_index):
    requested_index = int(requested_index)
    if requested_index < 0 or VISIBLE_GPU_COUNT <= 0:
        return requested_index
    if requested_index >= VISIBLE_GPU_COUNT:
        print(f"{name}: requested cuda:{requested_index}, using cuda:0 because only {VISIBLE_GPU_COUNT} CUDA device(s) are visible")
        return 0
    return requested_index


EFFECTIVE_WHISPER_DEVICE_INDEX = _clamp_device_index("WHISPER_DEVICE_INDEX", WHISPER_DEVICE_INDEX)
EFFECTIVE_ASR_MOE_DEVICE_INDEX = _clamp_device_index("ASR_MOE_DEVICE_INDEX", ASR_MOE_DEVICE_INDEX)
EFFECTIVE_PHOWHISPER_DEVICE_INDEX = _clamp_device_index("PHOWHISPER_DEVICE_INDEX", PHOWHISPER_DEVICE_INDEX)
EFFECTIVE_CTC_DEVICE_INDEX = _clamp_device_index("CTC_DEVICE_INDEX", CTC_DEVICE_INDEX)
EFFECTIVE_PANNS_DEVICE_INDEX = _clamp_device_index("PANNS_DEVICE_INDEX", PANNS_DEVICE_INDEX)
EFFECTIVE_DEMUCS_DEVICE_INDEX = _clamp_device_index("DEMUCS_DEVICE_INDEX", DEMUCS_DEVICE_INDEX)
EFFECTIVE_SEPREFORMER_DEVICE_INDEX = _clamp_device_index("SEPREFORMER_DEVICE_INDEX", SEPREFORMER_DEVICE_INDEX)
print("Effective device indices:", {
    "whisper": EFFECTIVE_WHISPER_DEVICE_INDEX,
    "asr_moe": EFFECTIVE_ASR_MOE_DEVICE_INDEX,
    "phowhisper": EFFECTIVE_PHOWHISPER_DEVICE_INDEX,
    "ctc": EFFECTIVE_CTC_DEVICE_INDEX,
    "panns": EFFECTIVE_PANNS_DEVICE_INDEX,
    "demucs": EFFECTIVE_DEMUCS_DEVICE_INDEX,
    "sepreformer": EFFECTIVE_SEPREFORMER_DEVICE_INDEX,
})

EFFECTIVE_COMPUTE_TYPE = COMPUTE_TYPE
if str(COMPUTE_TYPE).lower() == "float16":
    if VISIBLE_GPU_COUNT <= 0 or EFFECTIVE_WHISPER_DEVICE_INDEX < 0:
        EFFECTIVE_COMPUTE_TYPE = "float32"
    else:
        major, minor = torch.cuda.get_device_capability(EFFECTIVE_WHISPER_DEVICE_INDEX)
        gpu_name = torch.cuda.get_device_name(EFFECTIVE_WHISPER_DEVICE_INDEX)
        print("Whisper GPU capability:", gpu_name, f"sm_{major}{minor}")
        if major < 7:
            EFFECTIVE_COMPUTE_TYPE = "float32"
if EFFECTIVE_COMPUTE_TYPE != COMPUTE_TYPE:
    print(f"COMPUTE_TYPE: requested {COMPUTE_TYPE}, using {EFFECTIVE_COMPUTE_TYPE} for this CUDA backend")
else:
    print("COMPUTE_TYPE:", EFFECTIVE_COMPUTE_TYPE)

cmd = [
    "python", "main_original_ASR_MoE.py",
    "--input_folder_path", INPUT_DIR,
    "--config_path", "config.json",
    "--trace_run_dir", RUN_DIR,
    "--LLM", "case_2",
    "--dia3",
    "--merge_gap", "2.0",
    "--speaker-link-threshold", str(SPEAKER_LINK_THRESHOLD),
    "--sortformer-param",
    "--sortformer-pad-onset", "0.05",
    "--sortformer-pad-offset", "0.05",
    "--overlap_threshold", str(OVERLAP_THRESHOLD),
    "--whisper_arch", WHISPER_ARCH,
    "--asr_language", ASR_LANGUAGE,
    "--phowhisper_model_name", PHOWHISPER_MODEL_NAME,
    "--ctc_model_name", CTC_MODEL_NAME,
    "--compute_type", EFFECTIVE_COMPUTE_TYPE,
    "--threads", str(ASR_THREADS),
    "--whisper_device_index", str(EFFECTIVE_WHISPER_DEVICE_INDEX),
    "--asr_moe_device_index", str(EFFECTIVE_ASR_MOE_DEVICE_INDEX),
    "--phowhisper_device_index", str(EFFECTIVE_PHOWHISPER_DEVICE_INDEX),
    "--ctc_device_index", str(EFFECTIVE_CTC_DEVICE_INDEX),
    "--panns_device_index", str(EFFECTIVE_PANNS_DEVICE_INDEX),
    "--demucs_device_index", str(EFFECTIVE_DEMUCS_DEVICE_INDEX),
    "--sepreformer_device_index", str(EFFECTIVE_SEPREFORMER_DEVICE_INDEX),
    "--asr_micro_segment_seconds", str(ASR_MICRO_SEGMENT_SECONDS),
    "--asr_short_segment_seconds", str(ASR_SHORT_SEGMENT_SECONDS),
    "--asr_vi_agreement_threshold", str(ASR_VI_AGREEMENT_THRESHOLD),
    "--no-whisperx_word_timestamps",
    "--no-initprompt",
]
cmd.append("--ASRMoE" if ASR_MOE else "--no-ASRMoE")
cmd.append("--demucs" if RUN_DEMUCS else "--no-demucs")
cmd.append("--sepreformer" if RUN_SEPREFORMER else "--no-sepreformer")
cmd.append("--asr_quality_guard" if ASR_QUALITY_GUARD else "--no-asr_quality_guard")

run_logged(cmd, "18_main_full_pipeline.log", cwd="/kaggle/working/sommelier/podcast-pipeline", env=env, tail=80)


## 9. Gom output về run_full

In [ ]:
import json
import shutil
from pathlib import Path
import pandas as pd
from IPython.display import display

final_dir = Path(FINAL_DIR)
data_audio_dir = Path(DATA_AUDIO_DIR)
final_dir.mkdir(parents=True, exist_ok=True)
data_audio_dir.mkdir(parents=True, exist_ok=True)

json_files = sorted(Path(INPUT_DIR).glob("_final/**/*.json"), key=lambda p: p.stat().st_mtime)
if not json_files:
    raise FileNotFoundError(f"Không tìm thấy JSON output trong {Path(INPUT_DIR) / '_final'}")

source_json = json_files[-1]
source_segments_dir = source_json.with_suffix("")
print("Source JSON:", source_json)
print("Source MP3 dir:", source_segments_dir)

for old_mp3 in data_audio_dir.glob("*.mp3"):
    old_mp3.unlink()

source_mp3s = sorted(source_segments_dir.glob("*.mp3")) if source_segments_dir.exists() else []
for mp3 in source_mp3s:
    shutil.copy2(mp3, data_audio_dir / mp3.name)

with open(source_json, "r", encoding="utf-8") as f:
    data = json.load(f)
segments = data.get("segments", [])
for idx, segment in enumerate(segments):
    if idx < len(source_mp3s):
        segment["audio_file"] = f"data_audio/{source_mp3s[idx].name}"
    segment.setdefault("index", f"{idx:05d}")

metadata = data.setdefault("metadata", {})
metadata["source_branch"] = BRANCH
metadata["source_commit"] = subprocess.check_output(["git", "log", "-1", "--format=%H"], cwd="/kaggle/working/sommelier", text=True).strip()
metadata["source_json"] = str(source_json)
metadata["copied_mp3_count"] = len(source_mp3s)

final_json = final_dir / "data_audio.json"
raw_json = final_dir / "data_audio.raw.json"
with open(final_json, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)
shutil.copy2(source_json, raw_json)

print("Final JSON:", final_json)
print("Raw JSON copy:", raw_json)
print("Copied MP3:", len(source_mp3s))

rows = []
for seg in segments[:20]:
    rows.append({
        "index": seg.get("index"),
        "start": seg.get("start"),
        "end": seg.get("end"),
        "speaker": seg.get("speaker"),
        "text": seg.get("text"),
        "audio_file": seg.get("audio_file"),
    })
display(pd.DataFrame(rows))


## 10. Review output

In [ ]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import Audio, display

final_json = Path(FINAL_DIR) / "data_audio.json"
data = json.loads(final_json.read_text(encoding="utf-8"))
segments = data.get("segments", [])
print("Segments:", len(segments))
print("Metadata:", data.get("metadata", {}))

df = pd.DataFrame(segments)
cols = [c for c in ["index", "start", "end", "speaker", "text", "text_whisper", "text_phowhisper", "text_chunkformer", "audio_file"] if c in df.columns]
display(df[cols].head(20))

for i, seg in enumerate(segments[:5]):
    audio_file = seg.get("audio_file")
    if not audio_file:
        continue
    path = Path(FINAL_DIR) / audio_file
    print("\n--- segment", i, "---")
    print(seg.get("speaker"), seg.get("start"), seg.get("end"), seg.get("text", ""))
    print(path)
    if path.exists():
        display(Audio(str(path)))


## 11. Zip artifact thật để download

File zip chứa nguyên thư mục `run_full`: các artifact thật từng stage, final JSON/MP3/WAV/log. Notebook không tạo HTML; sau khi tải zip về, chạy script Python local để tạo HTML khi cần.


In [ ]:
import shutil
from pathlib import Path
from IPython.display import FileLink, display

run_dir = Path(RUN_DIR)
zip_base = Path("/kaggle/working/run_full_download")

required_outputs = [
    Path(RUN_DIR) / "01_diarization" / "diarization.json",
    Path(RUN_DIR) / "02_music_clean" / "segment_flags.json",
    Path(RUN_DIR) / "02_music_clean" / "cleaned_audio.wav",
    Path(RUN_DIR) / "03_overlap" / "segments.json",
    Path(RUN_DIR) / "04_asr" / "transcript.json",
    Path(FINAL_DIR) / "data_audio.json",
]
missing_outputs = [str(path) for path in required_outputs if not path.exists()]
if missing_outputs:
    raise FileNotFoundError("Thiếu output thật trước khi zip:\n" + "\n".join(missing_outputs))

zip_path = shutil.make_archive(
    base_name=str(zip_base),
    format="zip",
    root_dir=str(run_dir.parent),
    base_dir=run_dir.name,
)

print("Created:", zip_path)
print("Included run_full outputs:")
for path in required_outputs:
    print("-", path.relative_to(run_dir))
display(FileLink(zip_path))
